# 1) Crystals Breakdown

This notebook inspects the SCOPE Systems and retrieves statistics about HS and LS crystals.

**Important:** Notebooks must be executed in order. You're expected to run notebook 0 first.

In [1]:
import csv
import os
import sys

import scope

sys.path.insert(0, os.path.abspath('..'))
from local_functions import *

## User Configuration

In [ ]:
data_path     = "/Users/sergivela/Documents/SCOPE/ATLAS/Data/"                                                          # Update with your path
analysis_path = "/Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/1-TMC_Thermodynamics/"                  # Update with your path
env_path      = "/Users/sergivela/Documents/SCOPE/ATLAS/Data/scope_env_atlas.npy"                                       # Update with your path
figures_path  = "/Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/1-TMC_Thermodynamics/Figures/"          # Update with your path
save_figures  = True

if not save_figures:
    figures_path = None

# Normalizes the configured strings and defines the analysis output folder.
data_path     = os.path.abspath(os.path.expanduser(data_path)) + os.sep
env_path      = os.path.abspath(os.path.expanduser(env_path))
analysis_path = os.path.abspath(os.path.expanduser(analysis_path)) + os.sep
outputs_path  = analysis_path + "Outputs/"

# Checks SCOPE version
expected_scope_version = "0.9.6"
installed_scope_version = getattr(scope, "__version__", None)
if installed_scope_version != expected_scope_version:
    raise RuntimeError(f"This analysis requires SCOPE {expected_scope_version}; the active environment provides {installed_scope_version!r}.")

# These folders are required, together with the system-dedicated folders inside.
required_data_paths = [data_path + "2-Systems/", data_path + "3-Computations/"]
missing_paths       = [path for path in required_data_paths if not os.path.isdir(path)]
if missing_paths:
    raise FileNotFoundError(f"Missing required SCO ATLAS data directories: {missing_paths}")
if not os.path.isfile(env_path):
    raise FileNotFoundError(f"SCOPE environment file not found: {env_path}")
os.makedirs(outputs_path, exist_ok=True)

# Loads Environment
env = scope.load_binary(env_path)

# Paths of additional files
finished_csv_path      = outputs_path + "finished_systems.csv"
not_finished_csv_path  = outputs_path + "not_finished_systems.csv"
breakdown_csv1_path    = outputs_path + "breakdown_finished_crystals.csv"
breakdown_csv2_path    = outputs_path + "breakdown_non_finished_crystals.csv"

print(f"SCOPE version:      {installed_scope_version}")
print(f"Data directory:     {data_path}")
print(f"SCOPE environment:  {env_path}")
print(f"Analysis outputs:   {outputs_path}")
print(f"Manuscript Figures: {figures_path}")

SCOPE version:      0.9.6
Data directory:     /Users/sergivela/Documents/SCOPE/Database_SCO/Data/
SCOPE environment:  /Users/sergivela/Documents/SCOPE/Database_SCO/Data/scope_env_atlas.npy
Analysis outputs:   /Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/1-TMC_Thermodynamics/Outputs/
Manuscript Figures: /Users/sergivela/Documents/SCOPE/Sub-Projects/1-ATLAS/Analysis/1-TMC_Thermodynamics/Figures/


## Loads Systems

### All systems and Finished ones

In [3]:

finished_systems = []
not_finished = []
with open(finished_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sys = scope.load_binary(f"{data_path}/{row['path']}")
        sys.set_paths_from_environment(env)
        finished_systems.append(sys)

with open(not_finished_csv_path, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        sys = scope.load_binary(f"{data_path}/{row['path']}")
        sys.set_paths_from_environment(env)
        not_finished.append(sys)

print("Finished Systems:", len(finished_systems))
print("Not Finished Systems:", len(not_finished))

Finished Systems: 723
Not Finished Systems: 18


## Breakdown of Crystals

In [4]:
overwrite                   = True
breakdown_finished_rows     = []

In [5]:
num_crystals_HS = 0
num_crystals_LS = 0
num_crystals_other = 0
highest = 0

for sys in finished_systems:
    relative_path       = f"2-Systems/{sys.refcode}/{sys.refcode}.npy"
    # Counts number of original crystals
    ncrystals          = 0
    num_crystals_HS    = 0
    num_crystals_LS    = 0
    num_crystals_other = 0
    temp_HS_diff       = []
    temp_LS_diff       = []
    for sou in sys.sources:
        if sou.object_type == "cell" and "ref" not in sou.name:
            ncrystals += 1
            if hasattr(sou,"phase"):
                if   sou.phase == "HS": 
                    num_crystals_HS    += 1
                    temp_HS_diff.append(sou.cif.diff_temp)
                elif sou.phase == "LS": 
                    num_crystals_LS    += 1
                    temp_LS_diff.append(sou.cif.diff_temp)
                else:                   
                    num_crystals_other += 1
    # Checks if maximum
    if ncrystals > highest: 
        highest = ncrystals
        highest_refcode = sys.refcode
    breakdown_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "status": "finished", "ncrystals": ncrystals, "n_HS": num_crystals_HS, "n_LS": num_crystals_LS, "n_other": num_crystals_other, "temp_HS_diff": temp_HS_diff, "temp_LS_diff": temp_LS_diff}) 

with open(breakdown_csv1_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["refcode", "path", "status", "ncrystals", "n_HS", "n_LS", "n_other", "temp_HS_diff", "temp_LS_diff"])
    writer.writeheader()
    writer.writerows(breakdown_finished_rows)

print(f"Number of finished Systems:", count_csv_entries(breakdown_csv1_path, "status", "finished"))
print(f"Number of Crystals        :", sum_csv_entries(breakdown_csv1_path, "ncrystals"))
print(f"Number of HS Crystals     :", sum_csv_entries(breakdown_csv1_path, "n_HS"))
print(f"Number of LS Crystals     :", sum_csv_entries(breakdown_csv1_path, "n_LS"))
print(f"Unknown Crystal Phase     :", sum_csv_entries(breakdown_csv1_path, "n_other"))

Number of finished Systems: 723
Number of Crystals        : 1148
Number of HS Crystals     : 473
Number of LS Crystals     : 624
Unknown Crystal Phase     : 51


In [6]:
breakdown_non_finished_rows = []
highest = 0

for sys in not_finished:
    relative_path       = f"2-Systems/{sys.refcode}/{sys.refcode}.npy"
    # Counts number of original crystals
    ncrystals          = 0
    num_crystals_HS    = 0
    num_crystals_LS    = 0
    num_crystals_other = 0
    temp_HS_diff       = []
    temp_LS_diff       = []
    for sou in sys.sources:
        if sou.object_type == "cell" and "ref" not in sou.name:
            ncrystals += 1
            if hasattr(sou,"phase"):
                if   sou.phase == "HS": 
                    num_crystals_HS    += 1
                    temp_HS_diff.append(sou.cif.diff_temp)
                elif sou.phase == "LS": 
                    num_crystals_LS    += 1
                    temp_LS_diff.append(sou.cif.diff_temp)
                else:                   num_crystals_other += 1
    # Checks if maximum
    if ncrystals > highest: 
        highest = ncrystals
        highest_refcode = sys.refcode
    breakdown_non_finished_rows.append({"refcode": sys.refcode_wo_digits, "path": relative_path, "status": "not_finished", "ncrystals": ncrystals, "n_HS": num_crystals_HS, "n_LS": num_crystals_LS, "n_other": num_crystals_other, "temp_HS_diff": temp_HS_diff, "temp_LS_diff": temp_LS_diff}) 

with open(breakdown_csv2_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["refcode", "path", "status", "ncrystals", "n_HS", "n_LS", "n_other", "temp_HS_diff", "temp_LS_diff"])
    writer.writeheader()
    writer.writerows(breakdown_non_finished_rows)

print(f"Number of non finished Systems:", count_csv_entries(breakdown_csv2_path, "status", "not_finished"))
print(f"Number of Crystals            :", sum_csv_entries(breakdown_csv2_path, "ncrystals"))
print(f"Number of HS Crystals         :", sum_csv_entries(breakdown_csv2_path, "n_HS"))
print(f"Number of LS Crystals         :", sum_csv_entries(breakdown_csv2_path, "n_LS"))
print(f"Unknown Crystal Phase         :", sum_csv_entries(breakdown_csv2_path, "n_other"))

Number of non finished Systems: 18
Number of Crystals            : 21
Number of HS Crystals         : 4
Number of LS Crystals         : 16
Unknown Crystal Phase         : 1


## Systems with Both HS and LS crystals

In [7]:
count = 0
for item in filter_rows(breakdown_csv1_path, lambda row: int(row["n_HS"]) > 0 and int(row["n_LS"]) > 0):
    print(f"{item['refcode']:<10} {item['n_HS']:<10} {item['n_LS']:<10} {item['temp_HS_diff']:<40} {item['temp_LS_diff']:<40}")
    count += 1
print(count)

ABITEM     1          1          [240.0]                                  [104.0]                                 
ACAHUJ     1          1          [292.0]                                  [116.0]                                 
ADAQUU     1          1          [448.0]                                  [150.0]                                 
AFANEB     1          1          [200.0]                                  [90.0]                                  
AFOYUS     2          1          [293.0, 353.0]                           [123.0]                                 
AKENAF     1          1          [296.2]                                  [88.2]                                  
ATAVEY     1          1          [250.0]                                  [100.0]                                 
BAKGUR     1          1          [296.1]                                  [110.1]                                 
BAXFIS     2          2          [353.0, 338.0]                           [296.0